In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.fft as fft
import wandb
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
harm_X = np.load("/content/drive/MyDrive/128x128/harmonics_lid_driven_cavity_X.npz")['data'][:500]
harm_Y = np.load("/content/drive/MyDrive/128x128/harmonics_lid_driven_cavity_Y.npz")['data'][:500]

nurbs_X = np.load("/content/drive/MyDrive/128x128/nurbs_lid_driven_cavity_X.npz")["data"][:500]
nurbs_Y = np.load("/content/drive/MyDrive/128x128/nurbs_lid_driven_cavity_Y.npz")["data"][:500]

skeleton_X = np.load("/content/drive/MyDrive/128x128/skelneton_lid_driven_cavity_X.npz")["data"][:500]
skeleton_Y = np.load("/content/drive/MyDrive/128x128/skelneton_lid_driven_cavity_Y.npz")["data"][:500]

print("Harmonics X:", harm_X.shape)
print("Harmonics Y:", harm_Y.shape)
print("NURBS X:", nurbs_X.shape)
print("NURBS Y:", nurbs_Y.shape)
print("Skeleton X:", skeleton_X.shape)
print("Skeleton Y:", skeleton_Y.shape)

Harmonics X: (500, 3, 128, 128)
Harmonics Y: (500, 4, 128, 128)
NURBS X: (500, 3, 128, 128)
NURBS Y: (500, 4, 128, 128)
Skeleton X: (500, 3, 128, 128)
Skeleton Y: (500, 4, 128, 128)


In [ ]:
import torch.nn.functional as F

def downsample(arr, size=64):
    t = torch.tensor(arr).float()
    return F.interpolate(t, size=(size, size), mode='bilinear', align_corners=False).numpy()

harm_X     = downsample(harm_X)
harm_Y     = downsample(harm_Y)
nurbs_X    = downsample(nurbs_X)
nurbs_Y    = downsample(nurbs_Y)
skeleton_X = downsample(skeleton_X)
skeleton_Y = downsample(skeleton_Y)

print("After downsampling:")
print("harm_X:", harm_X.shape)  # (500, 3, 64, 64)

After downsampling:
harm_X: (500, 3, 64, 64)


Combining the Input fields and Keeping only 3 channels of the Output

In [ ]:
X = np.concatenate([harm_X, nurbs_X, skeleton_X], axis=0)
Y = np.concatenate([harm_Y, nurbs_Y, skeleton_Y], axis=0)

# Keeping only u, v, p
Y = Y[:, :3]

print("X:", X.shape)  # (1500, 3, 64, 64)
print("Y:", Y.shape)  # (1500, 3, 64, 64)

X: (1500, 3, 64, 64)
Y: (1500, 3, 64, 64)


### Converting to tensors and normalizing


In [ ]:
X = torch.tensor(X).float()
Y = torch.tensor(Y).float()

X_mean = X.mean(dim=(0,2,3), keepdim=True)
X_std  = X.std(dim=(0,2,3), keepdim=True) + 1e-6
X = (X - X_mean) / X_std

Y_mean = Y.mean(dim=(0,2,3), keepdim=True)
Y_std  = Y.std(dim=(0,2,3), keepdim=True) + 1e-6
Y = (Y - Y_mean) / Y_std

### Train test split

In [ ]:
train_size = int(0.8 * X.shape[0])

X_train = X[:train_size]
Y_train = Y[:train_size]

X_test = X[train_size:]
Y_test = Y[train_size:]

### Adding Coordinate Grid
FNO works better when given coordinates.

In [ ]:
def add_grid(x):
    batchsize, _, H, W = x.shape
    gridx = torch.linspace(0, 1, W)
    gridy = torch.linspace(0, 1, H)
    gridx, gridy = torch.meshgrid(gridx, gridy, indexing='xy')

    gridx = gridx.unsqueeze(0).unsqueeze(0).repeat(batchsize, 1, 1, 1)
    gridy = gridy.unsqueeze(0).unsqueeze(0).repeat(batchsize, 1, 1, 1)

    return torch.cat((x, gridx, gridy), dim=1)

X_train = add_grid(X_train)
X_test  = add_grid(X_test)

print(X_train.shape)  # (1200, 5, 64, 64)
print(X_test.shape)   # (300, 5, 64, 64)

torch.Size([1200, 5, 64, 64])
torch.Size([300, 5, 64, 64])


# Defining FNO

In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes = modes
        self.scale = 1 / (in_channels * out_channels)

        self.weights1 = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, modes, modes, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, modes, modes, dtype=torch.cfloat))

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft = fft.rfft2(x)

        out_ft = torch.zeros(B, self.out_channels, H, W // 2 + 1, dtype=torch.cfloat, device=x.device)

        out_ft[:, :, :self.modes, :self.modes] = torch.einsum(
            "bixy,ioxy->boxy",
            x_ft[:, :, :self.modes, :self.modes],
            self.weights1
        )

        out_ft[:, :, -self.modes:, :self.modes] = torch.einsum(
            "bixy,ioxy->boxy",
            x_ft[:, :, -self.modes:, :self.modes],
            self.weights2
        )

        x = fft.irfft2(out_ft, s=(H, W))
        return x


class FNO2d(nn.Module):
    def __init__(self, modes=12, width=64):
        super().__init__()
        self.fc0 = nn.Conv2d(5, width, 1)

        self.conv1 = SpectralConv2d(width, width, modes)
        self.conv2 = SpectralConv2d(width, width, modes)
        self.conv3 = SpectralConv2d(width, width, modes)
        self.conv4 = SpectralConv2d(width, width, modes)

        self.w1 = nn.Conv2d(width, width, 1)
        self.w2 = nn.Conv2d(width, width, 1)
        self.w3 = nn.Conv2d(width, width, 1)
        self.w4 = nn.Conv2d(width, width, 1)

        self.fc1 = nn.Conv2d(width, 128, 1)
        self.fc2 = nn.Conv2d(128, 3, 1)

    def forward(self, x):
        x = self.fc0(x)

        x = torch.relu(self.conv1(x) + self.w1(x))
        x = torch.relu(self.conv2(x) + self.w2(x))
        x = torch.relu(self.conv3(x) + self.w3(x))
        x = torch.relu(self.conv4(x) + self.w4(x))

        x = torch.relu(self.fc1(x))
        return self.fc2(x)

Adding Relative L2 loss, better than MSE for operator learning

In [ ]:
def relative_l2(pred, target):
    diff_norm   = torch.norm(pred.reshape(pred.shape[0], -1) - target.reshape(target.shape[0], -1), dim=1)
    target_norm = torch.norm(target.reshape(target.shape[0], -1), dim=1)
    return (diff_norm / target_norm).mean()

# Configuring W&B

In [ ]:
import wandb
import os
os.environ['WANDB_API_KEY']= "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"

In [ ]:
wandb.init(
    project="ppt-fno",
    name="run3-64x64-1500samples",
    config={
        "epochs": 50,
        "batch_size": 16,
        "learning_rate": 1e-3,
        "modes": 12,
        "width": 64,
        "grid_size": 64,
        "samples_per_geometry": 500,
        "total_samples": 1500
    }
)

In [ ]:
epochs        = 50
batch_size    = 16
learning_rate = 1e-3

device    = "cuda" if torch.cuda.is_available() else "cpu"
model     = FNO2d(modes=12, width=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn   = nn.MSELoss()

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test,  Y_test),  batch_size=batch_size, shuffle=False)

for epoch in range(epochs):

    # --- TRAINING ---
    model.train()
    train_loss_epoch = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        pred = model(xb)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_epoch += loss.item()

    train_loss_epoch /= len(train_loader)

    # --- TESTING ---
    model.eval()
    test_loss_epoch = 0.0
    rel_l2_epoch    = 0.0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)

            pred = model(xb)
            test_loss_epoch += loss_fn(pred, yb).item()
            rel_l2_epoch    += relative_l2(pred, yb).item()

    test_loss_epoch /= len(test_loader)
    rel_l2_epoch    /= len(test_loader)

    # --- LOGGING ---
    wandb.log({
        "epoch":            epoch,
        "train_loss":       train_loss_epoch,
        "test_loss":        test_loss_epoch,
        "relative_l2_error": rel_l2_epoch
    })

    print(f"Epoch {epoch:03d}/{epochs} | Train MSE: {train_loss_epoch:.6f} | Test MSE: {test_loss_epoch:.6f} | Rel L2: {rel_l2_epoch:.6f}")

wandb.finish()

Epoch 000/50 | Train MSE: 0.410267 | Test MSE: 0.179776 | Rel L2: 0.328226
Epoch 001/50 | Train MSE: 0.131394 | Test MSE: 0.089356 | Rel L2: 0.262951
Epoch 002/50 | Train MSE: 0.090204 | Test MSE: 0.093558 | Rel L2: 0.239659
Epoch 003/50 | Train MSE: 0.049938 | Test MSE: 0.166540 | Rel L2: 0.239524
Epoch 004/50 | Train MSE: 0.040418 | Test MSE: 0.028844 | Rel L2: 0.149668
Epoch 005/50 | Train MSE: 0.026887 | Test MSE: 0.023441 | Rel L2: 0.136553
Epoch 006/50 | Train MSE: 0.014169 | Test MSE: 0.018688 | Rel L2: 0.122940
Epoch 007/50 | Train MSE: 0.020566 | Test MSE: 0.065704 | Rel L2: 0.182302
Epoch 008/50 | Train MSE: 0.041123 | Test MSE: 0.027533 | Rel L2: 0.139513
Epoch 009/50 | Train MSE: 0.008379 | Test MSE: 0.015665 | Rel L2: 0.112632
Epoch 010/50 | Train MSE: 0.013161 | Test MSE: 0.032513 | Rel L2: 0.138572
Epoch 011/50 | Train MSE: 0.011427 | Test MSE: 0.050564 | Rel L2: 0.166939
Epoch 012/50 | Train MSE: 0.014706 | Test MSE: 0.015282 | Rel L2: 0.108658
Epoch 013/50 | Train MSE:

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
relative_l2_error,█▆▅▅▃▂▄▂▂▂▂▁▂▂▂▁▁▁▁▂▁▁▁▁▁▂▁▁▁▂▁▁▁▁▂▁▁▁▁▁
test_loss,█▄▄▇▂▁▃▂▁▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▂▁▁▁▁
train_loss,█▃▃▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,49
relative_l2_error,0.09569
test_loss,0.01173
train_loss,0.00269
